# 从结果输出四联图与单图

本例使用 0.1.1 候选源码。默认从内置合成 `P_ADO` 生成图件。要绘制自己的结果，把 `result_dir` 改为包含 `manifest.json` 和 `density.npz` 的目录，并设置图件目录和文件名前缀。图和统计与 GUI 共用核心和绘图 API。

Export the overview and four panels from the synthetic example or your own completed result snapshot.

In [ ]:
%matplotlib inline
from pathlib import Path

from multipolepdm.io.export import load_result
from multipolepdm.io.input import example_data, job_from_dict
from multipolepdm.jobs.workers import compute_job
from multipolepdm.pdm.statistics import summarize_density
from multipolepdm.plotting import make_preview, save_overview, save_panel

result_dir = None  # Example: Path('pdm-results/my-result')
output = Path('generated_figures')
stem = 'synthetic_P_ADO'

if result_dir is None:
    job = job_from_dict(example_data('P_ADO'))
    result, summary = compute_job(job)
else:
    result = load_result(result_dir)
    summary = summarize_density(result)

print(result.job.mode, result.job.label)
print('Source spin I =', result.response.reference_spin)
for warning in result.warnings:
    print(warning)

四联图保留响应全域、混合角 π 刻度及 σ/I 的源态定义。C、D 图用蓝色分段表示 68% HDI，用橙色标记等尾分位数，用灰色标记全分布的均值与标准差。C 图的转换报告从 `Summary.tan_angle_statistics` 读取，显式标作tan{mode[arctan(δ)]}、tan{HDI68%[arctan(δ)]}、tan{mean[arctan(δ)]}；tan(mu±std)只称转换端点。`Summary.delta_pdf`另存真正的δ密度，不能沿用角HDI水平阈值。

In [ ]:
output.mkdir(parents=True, exist_ok=True)
save_overview(result, summary, output / f'{stem}_overview.pdf')
save_overview(result, summary, output / f'{stem}_overview.png')
for panel in 'abcd':
    save_panel(result, summary, panel, output / f'{stem}_{panel}.svg')
print('Figures saved to', output.resolve())
make_preview(result, summary).figure